# Match Model + Tournament Simulation

Pipeline: load processed data -> engineer pre-match features -> train models -> evaluate -> simulate group and knockout stages -> export outputs.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
if PROJECT_ROOT.name != 'Project-3-WorldCup-2022-Predictions':
    PROJECT_ROOT = Path.cwd().resolve()

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

fixtures = pd.read_csv(PROCESSED_DIR / 'worldcup_2022_fixtures.csv', parse_dates=['date'])
teams = pd.read_csv(PROCESSED_DIR / 'worldcup_2022_teams.csv')
historical = pd.read_csv(PROCESSED_DIR / 'international_training_matches.csv', parse_dates=['date'])

fixtures.head(), historical.head()

In [ ]:
def match_points(gf: int, ga: int) -> int:
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0

def build_long_history(matches: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for r in matches.sort_values('date').itertuples(index=False):
        rows.append({
            'date': r.date,
            'team': r.home_team,
            'opponent': r.away_team,
            'gf': int(r.home_goals),
            'ga': int(r.away_goals),
            'gd': int(r.home_goals - r.away_goals),
            'points': match_points(int(r.home_goals), int(r.away_goals)),
            'neutral': bool(r.neutral)
        })
        rows.append({
            'date': r.date,
            'team': r.away_team,
            'opponent': r.home_team,
            'gf': int(r.away_goals),
            'ga': int(r.home_goals),
            'gd': int(r.away_goals - r.home_goals),
            'points': match_points(int(r.away_goals), int(r.home_goals)),
            'neutral': bool(r.neutral)
        })
    return pd.DataFrame(rows).sort_values(['team', 'date']).reset_index(drop=True)

long_history = build_long_history(historical)
long_history.head()

In [ ]:
def compute_team_snapshot(team: str, as_of_date: pd.Timestamp, window: int = 5) -> dict:
    hist = long_history[(long_history['team'] == team) & (long_history['date'] < as_of_date)].sort_values('date')
    recent = hist.tail(window)

    if recent.empty:
        return {
            'matches_played': 0.0,
            'ppg_last5': 1.0,
            'gf_last5': 1.0,
            'ga_last5': 1.0,
            'gd_last5': 0.0,
            'win_rate_last5': 0.33,
            'rating_proxy': 1500.0
        }

    wins = (recent['gf'] > recent['ga']).sum()
    rating_proxy = 1500.0 + 40.0 * recent['gd'].mean() + 3.0 * recent['points'].mean()

    return {
        'matches_played': float(len(hist)),
        'ppg_last5': float(recent['points'].mean()),
        'gf_last5': float(recent['gf'].mean()),
        'ga_last5': float(recent['ga'].mean()),
        'gd_last5': float(recent['gd'].mean()),
        'win_rate_last5': float(wins / len(recent)),
        'rating_proxy': float(rating_proxy)
    }

compute_team_snapshot('Argentina', pd.Timestamp('2022-11-20'))

In [ ]:
def build_feature_row(home_team: str, away_team: str, date: pd.Timestamp, neutral: bool) -> dict:
    h = compute_team_snapshot(home_team, date)
    a = compute_team_snapshot(away_team, date)

    row = {'neutral': int(bool(neutral))}
    for k, v in h.items():
        row[f'home_{k}'] = v
    for k, v in a.items():
        row[f'away_{k}'] = v

    for k in h.keys():
        row[f'home_minus_away_{k}'] = row[f'home_{k}'] - row[f'away_{k}']

    return row

feature_preview = build_feature_row('France', 'Australia', pd.Timestamp('2022-11-22'), neutral=True)
pd.Series(feature_preview).head(10)

In [ ]:
training_matches = historical.sort_values('date').reset_index(drop=True).copy()

X_rows, y_rows, date_rows = [], [], []
for r in training_matches.itertuples(index=False):
    feat = build_feature_row(r.home_team, r.away_team, r.date, r.neutral)
    X_rows.append(feat)
    y_rows.append(r.result)
    date_rows.append(r.date)

X = pd.DataFrame(X_rows)
y = pd.Series(y_rows, name='result')
event_dates = pd.Series(date_rows, name='date')

X.shape, y.value_counts()

In [ ]:
label_to_int = {'H': 0, 'D': 1, 'A': 2}
int_to_label = {v: k for k, v in label_to_int.items()}
y_enc = y.map(label_to_int)

assert y_enc.notna().all(), 'Target encoding failed for some rows.'
y_enc.value_counts()

In [ ]:
split_date = event_dates.quantile(0.8)
train_mask = event_dates <= split_date
valid_mask = event_dates > split_date

X_train, X_valid = X.loc[train_mask], X.loc[valid_mask]
y_train, y_valid = y_enc.loc[train_mask], y_enc.loc[valid_mask]

print('Split date:', split_date.date())
print('Train size:', len(X_train), '| Valid size:', len(X_valid))

In [ ]:
majority_class = y_train.mode().iloc[0]
baseline_pred = np.full(shape=len(y_valid), fill_value=majority_class)

baseline_metrics = {
    'model': 'MajorityClassBaseline',
    'accuracy': float(accuracy_score(y_valid, baseline_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_valid, baseline_pred))
}
baseline_metrics

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    random_state=42,
    min_samples_leaf=3,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=3000, multi_class='multinomial', random_state=42))
])
lr_model.fit(X_train, y_train)

In [ ]:
def evaluate_model(model, model_name: str, X_val: pd.DataFrame, y_val: pd.Series) -> tuple:
    pred = model.predict(X_val)
    metrics = {
        'model': model_name,
        'accuracy': float(accuracy_score(y_val, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_val, pred))
    }
    cm = confusion_matrix(y_val, pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(cm, index=['actual_H', 'actual_D', 'actual_A'], columns=['pred_H', 'pred_D', 'pred_A'])
    return metrics, cm_df

rf_metrics, rf_cm = evaluate_model(rf_model, 'RandomForestClassifier', X_valid, y_valid)
lr_metrics, lr_cm = evaluate_model(lr_model, 'LogisticRegression', X_valid, y_valid)

display(pd.DataFrame([baseline_metrics, rf_metrics, lr_metrics]).sort_values('balanced_accuracy', ascending=False))
display(rf_cm)
display(lr_cm)

In [ ]:
candidates = [rf_metrics, lr_metrics]
best_name = max(candidates, key=lambda d: d['balanced_accuracy'])['model']
best_model = rf_model if best_name == 'RandomForestClassifier' else lr_model
print('Selected model by balanced_accuracy:', best_name)

In [ ]:
wc_feature_rows = []
for r in fixtures.sort_values('match_no').itertuples(index=False):
    feat = build_feature_row(r.home_team, r.away_team, pd.to_datetime(r.date), True)
    feat.update({
        'match_no': r.match_no,
        'date': r.date,
        'stage': r.stage,
        'group': r.group,
        'home_team': r.home_team,
        'away_team': r.away_team
    })
    wc_feature_rows.append(feat)

wc_features = pd.DataFrame(wc_feature_rows).sort_values('match_no').reset_index(drop=True)
wc_features.head()

In [ ]:
model_feature_cols = [c for c in X.columns if c in wc_features.columns]
proba = best_model.predict_proba(wc_features[model_feature_cols])
class_order = list(best_model.classes_)

proba_df = pd.DataFrame(proba, columns=[f'class_{c}' for c in class_order])
for label, idx in label_to_int.items():
    col_name = f'class_{idx}'
    outcome_name = 'home_win' if label == 'H' else ('draw' if label == 'D' else 'away_win')
    wc_features[f'p_{outcome_name}'] = proba_df[col_name] if col_name in proba_df.columns else 0.0

wc_features['pred_class_int'] = best_model.predict(wc_features[model_feature_cols])
wc_features['pred_result'] = wc_features['pred_class_int'].map(int_to_label)

wc_features[['match_no', 'home_team', 'away_team', 'p_home_win', 'p_draw', 'p_away_win', 'pred_result']].head()

In [ ]:
group_matches = wc_features[wc_features['stage'] == 'Group'].copy()

def resolve_group_match(row: pd.Series, rng: np.random.Generator, mode: str = 'sample') -> tuple:
    probs = np.array([row['p_home_win'], row['p_draw'], row['p_away_win']], dtype=float)
    probs = np.clip(probs, 1e-9, None)
    probs = probs / probs.sum()
    outcomes = np.array(['H', 'D', 'A'])

    if mode == 'argmax':
        out = outcomes[int(np.argmax(probs))]
    else:
        out = rng.choice(outcomes, p=probs)

    if out == 'H':
        return out, 2, 1
    if out == 'A':
        return out, 1, 2
    return out, 1, 1

def rank_group_table(df: pd.DataFrame) -> pd.DataFrame:
    return df.sort_values(['points', 'gd', 'gf', 'team'], ascending=[False, False, False, True]).reset_index(drop=True)

def simulate_group_stage(group_df: pd.DataFrame, seed: int = 42, mode: str = 'sample'):
    rng = np.random.default_rng(seed)
    standings = {}
    match_preds = []

    for group_name, gdf in group_df.groupby('group'):
        team_names = sorted(set(gdf['home_team']).union(set(gdf['away_team'])))
        table = pd.DataFrame({
            'team': team_names,
            'group': group_name,
            'points': 0, 'gf': 0, 'ga': 0, 'gd': 0,
            'wins': 0, 'draws': 0, 'losses': 0, 'played': 0
        })

        for row in gdf.sort_values('match_no').itertuples(index=False):
            out, h_g, a_g = resolve_group_match(pd.Series(row._asdict()), rng, mode=mode)

            h_idx = table.index[table['team'] == row.home_team][0]
            a_idx = table.index[table['team'] == row.away_team][0]

            table.loc[h_idx, ['played', 'gf', 'ga', 'gd']] += [1, h_g, a_g, h_g - a_g]
            table.loc[a_idx, ['played', 'gf', 'ga', 'gd']] += [1, a_g, h_g, a_g - h_g]

            if out == 'H':
                table.loc[h_idx, ['points', 'wins']] += [3, 1]
                table.loc[a_idx, 'losses'] += 1
            elif out == 'A':
                table.loc[a_idx, ['points', 'wins']] += [3, 1]
                table.loc[h_idx, 'losses'] += 1
            else:
                table.loc[h_idx, ['points', 'draws']] += [1, 1]
                table.loc[a_idx, ['points', 'draws']] += [1, 1]

            match_preds.append({
                'match_no': int(row.match_no),
                'date': str(pd.to_datetime(row.date).date()),
                'stage': row.stage,
                'group': row.group,
                'home_team': row.home_team,
                'away_team': row.away_team,
                'p_home_win': float(row.p_home_win),
                'p_draw': float(row.p_draw),
                'p_away_win': float(row.p_away_win),
                'pred_result': out,
                'pred_home_goals': int(h_g),
                'pred_away_goals': int(a_g)
            })

        standings[group_name] = rank_group_table(table)

    return standings, pd.DataFrame(match_preds).sort_values('match_no').reset_index(drop=True)

group_tables, group_stage_predictions = simulate_group_stage(group_matches, seed=42, mode='sample')
display(group_stage_predictions.head())
for g in sorted(group_tables.keys()):
    display(group_tables[g])

In [ ]:
qualifiers = {}
for g, table in group_tables.items():
    qualifiers[f'{g}1'] = table.iloc[0]['team']
    qualifiers[f'{g}2'] = table.iloc[1]['team']

qualifiers

In [ ]:
r16_pairings = [
    ('R16-1', 'A1', 'B2'), ('R16-2', 'C1', 'D2'), ('R16-3', 'E1', 'F2'), ('R16-4', 'G1', 'H2'),
    ('R16-5', 'B1', 'A2'), ('R16-6', 'D1', 'C2'), ('R16-7', 'F1', 'E2'), ('R16-8', 'H1', 'G2')
]

def match_probability_row(team_a: str, team_b: str, as_of_date: pd.Timestamp) -> dict:
    feat = build_feature_row(team_a, team_b, as_of_date, True)
    feat_df = pd.DataFrame([feat])[model_feature_cols]
    probs = best_model.predict_proba(feat_df)[0]
    p_map = {int_to_label[c]: float(p) for c, p in zip(best_model.classes_, probs)}
    return {
        'p_home_win': p_map.get('H', 0.0),
        'p_draw': p_map.get('D', 0.0),
        'p_away_win': p_map.get('A', 0.0)
    }

def resolve_knockout_match(team_a: str, team_b: str, stage: str, as_of_date: pd.Timestamp, rng: np.random.Generator):
    probs = match_probability_row(team_a, team_b, as_of_date)
    pred = max(['H', 'D', 'A'], key=lambda o: probs['p_home_win'] if o == 'H' else probs['p_draw'] if o == 'D' else probs['p_away_win'])

    if pred == 'D':
        total = probs['p_home_win'] + probs['p_away_win']
        if total > 0:
            p_team_a = probs['p_home_win'] / total
        else:
            p_team_a = 0.5
        winner = team_a if rng.random() < p_team_a else team_b
        score = (1, 1)
    elif pred == 'H':
        winner = team_a
        score = (2, 1)
    else:
        winner = team_b
        score = (1, 2)

    return {
        'stage': stage,
        'date': str(as_of_date.date()),
        'home_team': team_a,
        'away_team': team_b,
        'pred_home_goals': score[0],
        'pred_away_goals': score[1],
        'p_home_win': probs['p_home_win'],
        'p_draw': probs['p_draw'],
        'p_away_win': probs['p_away_win'],
        'pred_result': pred,
        'winner': winner
    }

In [ ]:
rng = np.random.default_rng(42)

knockout_rows = []

r16_winners = []
for i, (_, left_slot, right_slot) in enumerate(r16_pairings, start=1):
    team_a = qualifiers[left_slot]
    team_b = qualifiers[right_slot]
    result = resolve_knockout_match(team_a, team_b, 'R16', pd.Timestamp('2022-12-03') + pd.Timedelta(days=(i // 2)), rng)
    result['match_label'] = f'R16-{i}'
    knockout_rows.append(result)
    r16_winners.append(result['winner'])

qf_pairs = [(r16_winners[0], r16_winners[1]), (r16_winners[2], r16_winners[3]), (r16_winners[4], r16_winners[5]), (r16_winners[6], r16_winners[7])]
qf_winners = []
for i, (team_a, team_b) in enumerate(qf_pairs, start=1):
    result = resolve_knockout_match(team_a, team_b, 'QF', pd.Timestamp('2022-12-09') + pd.Timedelta(days=(i // 3)), rng)
    result['match_label'] = f'QF-{i}'
    knockout_rows.append(result)
    qf_winners.append(result['winner'])

sf_pairs = [(qf_winners[0], qf_winners[1]), (qf_winners[2], qf_winners[3])]
sf_winners, sf_losers, semifinalists = [], [], []
for i, (team_a, team_b) in enumerate(sf_pairs, start=1):
    semifinalists.extend([team_a, team_b])
    result = resolve_knockout_match(team_a, team_b, 'SF', pd.Timestamp('2022-12-13') + pd.Timedelta(days=i - 1), rng)
    result['match_label'] = f'SF-{i}'
    knockout_rows.append(result)
    sf_winners.append(result['winner'])
    sf_losers.append(team_b if result['winner'] == team_a else team_a)

final_result = resolve_knockout_match(sf_winners[0], sf_winners[1], 'Final', pd.Timestamp('2022-12-18'), rng)
final_result['match_label'] = 'Final-1'
knockout_rows.append(final_result)

knockout_predictions = pd.DataFrame(knockout_rows)
champion = final_result['winner']
runner_up = sf_winners[1] if champion == sf_winners[0] else sf_winners[0]

display(knockout_predictions)
print('Champion:', champion)
print('Runner-up:', runner_up)

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

group_stage_predictions.to_csv(OUTPUTS_DIR / 'group_stage_predictions.csv', index=False)
knockout_predictions.to_csv(OUTPUTS_DIR / 'knockout_predictions.csv', index=False)

summary = {
    'champion': champion,
    'runner_up': runner_up,
    'semifinalists': sorted(set(semifinalists))
}

with open(OUTPUTS_DIR / 'tournament_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Exported outputs to', OUTPUTS_DIR)
summary

In [ ]:
assert len(fixtures) == 64, 'Fixtures count must be 64.'
assert len(group_tables) == 8, 'Group simulation must output 8 groups.'
assert all(len(tbl) == 4 for tbl in group_tables.values()), 'Each group must have 4 teams.'
assert (knockout_predictions['stage'] == 'R16').sum() == 8, 'R16 must have 8 matches.'
assert (knockout_predictions['stage'] == 'QF').sum() == 4, 'QF must have 4 matches.'
assert (knockout_predictions['stage'] == 'SF').sum() == 2, 'SF must have 2 matches.'
assert (knockout_predictions['stage'] == 'Final').sum() == 1, 'Final must have 1 match.'
assert all(k in summary for k in ['champion', 'runner_up']), 'Summary JSON keys missing.'
print('Minimal validation checklist passed.')

In [ ]:
print(f'Final champion prediction: {champion}')

## Assumptions And Limitations
- Features are compact proxies (recent form/goals/rating proxy), not full event-level features.
- Knockout draws use weighted winner selection from non-draw probabilities.
- Simulation outputs are seed-controlled and reproducible but still model-dependent.